# Experimento pareado: LSTM32 con entradas completas frente a solo telemetría — Colab CPU

Se entrena desde cero la mejor LSTM hasta ahora (LSTM32 de una capa, dropout 0,2, BCE con `pos_weight=2`, selección de checkpoint y umbral por F2; F2 de validación 0,46223 con semilla 42) dos veces en la misma sesión CPU:

| Hijo | Entradas |
|---|---|
| `lstm32_inputs_completos` (referencia) | `muestras_preprocesadas_v1_3`: 22 numéricas por vuelta, `Compound_asof`, `track_phase_asof` y contexto estático `Driver`, `Team`, `GranPremio`. |
| `lstm32_solo_telemetria` (candidata) | `muestras_preprocesadas_v1_3_telemetria_v1`: 15 numéricas (progreso, neumático, ritmo, posición, telemetría del coche y lluvia), las mismas dos categóricas dinámicas y **ningún contexto estático**. |

Se eliminan `Driver`, `Team`, `GranPremio` y el clima casi constante dentro de una carrera (`Weather_AirTemp`, `Weather_Humidity`, `Weather_Pressure`, `Weather_TrackTemp`) junto con el viento (`Weather_WindDirection_sin/cos`, `Weather_WindSpeed`). La rama estática de la arquitectura se mantiene, pero recibe un único marcador constante (ID reservado 1), así que no aporta información. Las estadísticas de las variables conservadas son exactamente las del preprocesador congelado v1_3: no hay reajuste.

Todo lo demás queda fijo: particiones, población elegible, etiquetas, W=8, máscaras, arquitectura, pérdida, Adam 0,001, semilla 42 y runtime CPU. El runner rechaza cualquier otra diferencia. El test 2025 permanece reservado.

## 1. Subir y verificar el paquete CPU

El paquete está en `runs/lstm_colab_telemetria_cpu_v1.zip`. Para regenerarlo desde la raíz del repositorio: `python scripts/empaquetar_telemetria.py --output runs/lstm_colab_telemetria_cpu_v1.zip`. Si lo regeneras, el SHA-256 cambia y debes actualizar `EXPECTED_PACKAGE_SHA256`. Sube el ZIP cuando la celda lo pida.

In [ ]:
from pathlib import Path
import hashlib, json, stat, zipfile
from google.colab import files

ROOT = Path('/content/f1-lstm-telemetria-cpu-v1')
PACKAGE = Path('/content/lstm_colab_telemetria_cpu_v1.zip')
EXPECTED_PACKAGE_SHA256 = '6a52519914b5d29210b2fefa05a491e5fad79a86f4f6a29ec4ba4209c59e8e9a'
if ROOT.exists(): raise FileExistsError('Usa una sesión limpia: /content/f1-lstm-telemetria-cpu-v1 ya existe.')
if not PACKAGE.is_file():
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError('Sube exactamente el ZIP CPU del par LSTM completas/telemetría.')
    name, content = next(iter(uploaded.items()))
    if Path(name).name != PACKAGE.name: raise ValueError(f'Se requiere {PACKAGE.name}.')
    PACKAGE.write_bytes(content)
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()
package_hash = sha256(PACKAGE)
print('SHA-256 del paquete:', package_hash)
if package_hash != EXPECTED_PACKAGE_SHA256: raise ValueError('El hash no coincide con el ZIP publicado.')
with zipfile.ZipFile(PACKAGE) as z:
    names = set()
    for m in z.infolist():
        p = Path(m.filename)
        if p.is_absolute() or '..' in p.parts or '\\' in m.filename or stat.S_ISLNK(m.external_attr >> 16): raise ValueError(f'Ruta ZIP no segura: {m.filename}')
        if m.filename in names: raise ValueError(f'Miembro duplicado: {m.filename}')
        names.add(m.filename)
    manifest = json.loads(z.read('colab_package_manifest.json'))
    if manifest.get('artifact_version') != 'lstm_colab_telemetry_cpu_inputs_v1' or manifest.get('test_arrays_included') is not False: raise ValueError('Contrato de paquete no admitido o contiene test.')
    if names != set(manifest['files_sha256']) | {'colab_package_manifest.json'}: raise ValueError('Los archivos del ZIP no coinciden con su manifiesto.')
    ROOT.mkdir(parents=True)
    z.extractall(ROOT)
for name, digest in manifest['files_sha256'].items():
    if sha256(ROOT / name) != digest: raise ValueError(f'SHA-256 incorrecto: {name}')
print('Manifiesto verificado:', len(manifest['files_sha256']), 'archivos; sin datos de prueba.')

## 2. Instalar dependencias y confirmar runtime CPU

Ambas configuraciones fijan `device=cpu`. Aunque Colab muestre una GPU, el entrenamiento corre en CPU. Cada hijo tardó unos 4–5 minutos en la pareja CPU anterior.

In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = Path('/content/f1-lstm-telemetria-cpu-v1')
if not (ROOT / 'requirements-colab-experiments.txt').is_file():
    raise RuntimeError('No encuentro el paquete extraído. Ejecuta primero la sección 1.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements-colab-experiments.txt')], check=True)
import torch
print('PyTorch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available(), '| dispositivo efectivo: CPU')

## 3. Revisar las entradas de cada hijo

Esta celda solo lee los esquemas congelados y la cuota de varianza dentro de carrera que justificó la selección (calculada en train elegible). No entrena ni lee 2025.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path('/content/f1-lstm-telemetria-cpu-v1')
sys.path.insert(0, str(ROOT / 'src'))
from f1_pitstop.rnn_data import read_json, verify_preprocessed
for role in ('ref', 'candidate'):
    cfg = read_json(ROOT / f'configs/experimento_lstm32_telemetria_{role}_cpu_v1.json')
    _, state = verify_preprocessed(ROOT / cfg['preprocessed_package'], cfg['preprocessed_manifest_sha256'])
    schema = state['schema']
    print(f"{role}: {cfg['preprocessed_package']}")
    print('  numéricas  :', len(schema['numeric_feature_names']), schema['numeric_feature_names'])
    print('  categóricas:', schema['categorical_feature_names'])
    print('  estáticas  :', schema['static_feature_names'])
report = read_json(ROOT / 'EDA/dataset/muestras_preprocesadas_v1_3_telemetria_v1/config.json')
print('\nEliminadas:', report['removed_static'] + report['removed_numeric'])

## 4. Entrenar la pareja en CPU

Primero la referencia con entradas completas (debería reproducir aproximadamente F2≈0,462 de la corrida `pos_weight=2` semilla 42) y luego la candidata solo telemetría. Ambas usan early stopping en validación a umbral 0,5, paciencia seis y desempate por BCE menor; después se elige el umbral por F2 en la cuadrícula congelada. Si falla, la celda imprime `status.json`.

In [ ]:
import json, subprocess, sys
from pathlib import Path
ROOT = Path('/content/f1-lstm-telemetria-cpu-v1')
RUN = ROOT / 'runs/lstm32_telemetria_f2_cpu_v1_seed42'
if RUN.exists() or RUN.with_suffix('.zip').exists(): raise FileExistsError('La salida ya existe. Usa una sesión nueva.')
result = subprocess.run(
    [sys.executable, 'scripts/entrenar_comparacion_telemetria.py',
     '--reference-config', 'configs/experimento_lstm32_telemetria_ref_cpu_v1.json',
     '--candidate-config', 'configs/experimento_lstm32_telemetria_candidate_cpu_v1.json',
     '--output', str(RUN.relative_to(ROOT))],
    cwd=ROOT, check=False,
)
if result.returncode != 0:
    status_path = RUN / 'status.json'
    if status_path.is_file(): print(json.dumps(json.loads(status_path.read_text()), indent=2))
    raise RuntimeError(f'La corrida falló con código {result.returncode}; revisa el estado antes de reintentar.')
print('Pareja completada:', RUN.with_suffix('.zip'))

## 5. Comparar resultados de validación 2024

In [ ]:
import json
import pandas as pd
from pathlib import Path
RUN = Path('/content/f1-lstm-telemetria-cpu-v1/runs/lstm32_telemetria_f2_cpu_v1_seed42')
summary = json.loads((RUN / 'pair_summary.json').read_text())
rows = {}
for role, label in (('reference', 'Entradas completas'), ('candidate', 'Solo telemetría')):
    child = summary[role]
    selected = child['validation']['selected_threshold']
    rows[label] = {'época': child['best_epoch'], 'umbral': child['threshold'], 'parámetros': child['parameter_count'],
                   'F2': selected['f2'], 'F1': selected['f1'], 'precisión': selected['precision'],
                   'recall': selected['recall'], 'AP': selected['average_precision'],
                   'FP': selected['fp'], 'FN': selected['fn'], 'BCE val': child['validation']['bce']}
table = pd.DataFrame(rows)
table['Δ (telemetría − completas)'] = table['Solo telemetría'] - table['Entradas completas']
display(table)
print('Variables eliminadas:', summary['changed_factor']['removed_static'] + summary['changed_factor']['removed_numeric'])

## 6. Descargar y devolver al proyecto

Descarga `lstm32_telemetria_f2_cpu_v1_seed42.zip` y guárdalo intacto en `runs/` del repositorio. Localmente se puede reproducir cada hijo con `scripts/evaluar_rnn.py`, pasando `--data EDA/dataset/muestras_preprocesadas_v1_3_telemetria_v1` para la candidata. Es una sola semilla: si la diferencia es pequeña, replica con 43 y 44 antes de concluir. El test 2025 no se utilizó.

In [ ]:
from google.colab import files
from pathlib import Path
files.download(str(Path('/content/f1-lstm-telemetria-cpu-v1/runs/lstm32_telemetria_f2_cpu_v1_seed42.zip')))